#1 — DataExplore MAIN TeamShared

In [1]:
#2 — SETUP: Install required packages in the current kernel
# Run this cell first if you encounter ModuleNotFoundError
%pip install seaborn opencv-contrib-python scikit-image plotly joblib imbalanced-learn tensorflow Pillow fidle -q 2>&1 | tail -1

Note: you may need to restart the kernel to use updated packages.


#3 — [Datascientest](https://datascientest.com/) project - Plant Vs Disease
---
+ Date 2022/10/

Team
----
+ Promo **Sep2022 DS Bootcamp**
+ Bertrand Chatelet (apprenant)
+ François Condominas (apprenant) <nav><a href="https://www.linkedin.com/in/francoiscondominas">LinkedIn</a> | <a href="https://github.com/fcondo56">GitHub</a></nav>
+ Jeremy Cuvelier (apprenant)
+ Zakary Saheb (mentor)
+ Romain Godet (chef de cohorte)
— Raw Data
---
+ source: [kaggle - new-plant-diseases-dataset](https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset?resource=download)
+ gitignore data/RawData
— Program structure
---
1. _Import_ **PACKAGES**
2. **MAIN** _(includes (calling) FUNCTIONS in a **sequence**)_
3. **Validated FUNCTIONS** _(all FUNCTIONS in one code cell)_

4. _others code cells for **new functions under debug** or **under approval**_



*Note:* <span style="color:green">_'Images'_</span> *environment variable set to* <span style="color:green">_.../train_</span> *folder...*

In [14]:
#4 — IMPORT PACKAGES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline                  
print('%matplotlib inline only for notebook...')
import seaborn as sns
import pprint                                       #pprint to make some output look nicer
#pp = pprint.PrettyPrinter(indent=4)
from collections import Counter
import os
from os import path
#to manage local files         
import cv2 as cv                                    #Open CV, need to powershell install w/ the following: pip install opencv-contrib-python
from PIL import Image                               #Pillow ou PIL (Python Imaging Library)
import glob                                         #globbing
import joblib
from skimage.io import imread
from skimage.transform import resize
import plotly.express as px                         #plotly for 3D xplot
from datetime import datetime

%matplotlib inline only for notebook...


In [25]:
#5 — MAIN PROGRAM SEQUENCE

from plant_utils import *

start_flag()
data_path = manage_CWD()
#EXECUTE the FUNCTION JUST BELOW TO INITIATE THE DF (with basic features)
df = raw_data_to_dataframe(data_path)               #can be executed once, up to you
df_to_csv(df, 'df_plant_new2')                      #export du DataFrame en CSV

end_flag()


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~~~~~~~~~~
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~~~~~~~~~~
      ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~  Plant Vs Disease  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~~~~~~~~~~
  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
... 16:56:12

---------
 * CWD * 
---------

⚠️  Variable d'environnement "Images" non définie.
   Utilisation du chemin relatif par défaut.
Current CWD: /mnt/d/business/memetics/cand/cand2026/IA/github-ref/PlantVSDisease/data/PlantVSDiseasesDataset/train
data_path: /mnt/d/business/memetics/cand/cand2026/IA/github-ref/PlantVSDisease/data/PlantVSDiseasesDataset/tr

In [26]:
#6 — LabelEncoder (sans charger les images)
# Les images sont chargées à la volée via ImageDataGenerator dans la cellule #7

from sklearn.preprocessing import LabelEncoder
import numpy as np

lz = 3  # RGB : 3 canaux

y = df["label_disease"]
y = y.apply(lambda x: "Nonehealthy" if not x.startswith('healthy') else "healthy")
print("Classes :", y.unique())
print(y.value_counts())

encoder = LabelEncoder()
y = encoder.fit_transform(y)
print("y shape:", y.shape)
print("Distribution des classes:", np.bincount(y))

# Ne PAS charger les images ici — trop de mémoire
print("✅ Labels encodés. Les images seront chargées à la volée dans les cellules suivantes.")

Classes : ['Nonehealthy' 'healthy']
label_disease
healthy        5
Nonehealthy    3
Name: count, dtype: int64
y shape: (8,)
Distribution des classes: [3 5]
✅ Labels encodés. Les images seront chargées à la volée dans les cellules suivantes.


In [28]:
#7 — train/test split + DataGenerator (charge les images à la volée)

import importlib
import plant_utils
importlib.reload(plant_utils)
from plant_utils import df_to_generator
from sklearn.model_selection import train_test_split
import numpy as np

# Split uniquement sur les indices (pas besoin de X en mémoire)
train_idx, test_idx, y_train, y_test = train_test_split(
    np.arange(len(df)), y, test_size=0.20, random_state=66, stratify=y
)

print(f"Train: {len(train_idx)}, Test: {len(test_idx)}")
print("Distribution y_train:", np.bincount(y_train))
print("Distribution y_test:", np.bincount(y_test))

BATCH_SIZE = 32
train_gen = df_to_generator(df, train_idx, y_train, batch_size=BATCH_SIZE)
test_gen = df_to_generator(df, test_idx, y_test, batch_size=BATCH_SIZE)

# Vérification : un batch
X_batch, y_batch = next(train_gen())
print("Batch X shape:", X_batch.shape)
print("Batch y shape:", y_batch.shape)
print("✅ DataGenerator prêt. Utiliser model.fit(train_gen(), ...) pour l'entraînement.")

Train: 6, Test: 2
Distribution y_train: [2 4]
Distribution y_test: [1 1]
Batch X shape: (6, 256, 256, 3)
Batch y shape: (6,)
✅ DataGenerator prêt. Utiliser model.fit(train_gen(), ...) pour l'entraînement.


In [29]:
#8 — Model definition

from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras.models import Model

if lz==1: 
    inputs = Input(shape = (256, 256,1), name = "Input")
elif lz==3:
    inputs = Input(shape = (256, 256,3), name = "Input")

first_layer = Conv2D(filters = 32,
                     kernel_size = (3, 3),
                     padding = 'valid',
                     activation = 'relu')

second_layer = MaxPooling2D(pool_size = (2, 2))
third_layer = Dropout(rate = 0.2)
fourth_layer = Conv2D(filters = 32,
                     kernel_size = (3, 3),
                     padding = 'valid',
                     activation = 'relu')
fifth_layer = MaxPooling2D(pool_size = (2, 2))
sixth_layer=Dropout(rate = 0.2)
seventh_layer = Flatten()
output_layer = Dense(units = 2,
                     activation='softmax')
x=first_layer(inputs)
x=second_layer(x)
x=third_layer(x)
x=fourth_layer(x)
x=fifth_layer(x)
x=sixth_layer(x)
x=seventh_layer(x)
outputs=output_layer(x)

model = Model(inputs = inputs, outputs = outputs)